# 🏗️ Landing Layer Architectural Design Decisions

---

This notebook fetches seismic data from the USGS API year by year, running in **reverse chronological order** (from 2026 back to 2000).

### 1. Decoupled Dual-Storage Architecture (Landing vs. Bronze)
> **The Decision:** 
> We implement a clear distinction between our **Landing Zone** and our **Bronze Table**. The raw, unaltered GeoJSON responses are saved as physical files (`.json`) inside a Unity Catalog Managed Volume (`/Volumes/cr_seismic_analysis/landing/seismic_data_raw/`). 

In [0]:
import requests
import json
import time 
import os
from datetime import datetime

# 1. Parameter configuration
CURRENT_YEAR = datetime.now().year
START_YEAR = 2000
END_YEAR = CURRENT_YEAR
BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
LANDING_VOLUME_PATH = f"/Volumes/cr_seismic_analysis/landing/seismic_data_raw/"

# Ensure the Managed Volume directory exists
os.makedirs(LANDING_VOLUME_PATH, exist_ok=True)

# 2. Reverse chronological ingestion loop to Managed Volume
for year in range(END_YEAR, START_YEAR - 1, -1):
    file_name = f"seismic_raw_{year}.json"
    dest_path = os.path.join(LANDING_VOLUME_PATH, file_name)
    
    # Skip API requests for static historical years if the file already exists in the Volume
    if year != CURRENT_YEAR and os.path.exists(dest_path):
        print(f"Year {year} already exists in Volume. Skipping API request.")
        continue
        
    print(f"Downloading GeoJSON from USGS API for year {year}...")
    
    params = {
        "format": "geojson",
        "starttime": f"{year}-01-01",
        "endtime": f"{year}-12-31",
        "minlatitude": 8.0,
        "maxlatitude": 11.5,
        "minlongitude": -86.0,
        "maxlongitude": -82.5
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        
        if response.status_code == 200:
            json_data = response.json()
            
            # Persist raw JSON payload to the Managed Volume (Landing)
            with open(dest_path, "w") as f:
                json.dump(json_data, f)
            print(f"-> Successfully saved: {file_name}")
            
        else:
            print(f"⚠️ HTTP error {response.status_code} encountered for year {year}.")
            
    except Exception as e:
        print(f"❌ Connection error occurred for year {year}: {str(e)}")
        
    print("Waiting 1.5 seconds to comply with API rate-limiting...")
    time.sleep(1.5)

In [0]:
%sql
-- Verification of the files in landing
USE CATALOG cr_seismic_analysis;

-- List files using the explicit string path
LIST '/Volumes/cr_seismic_analysis/landing/seismic_data_raw/';